In [4]:
#@title Install and import packages and auth Google sheets
%%capture
!pip install -q -U agno duckdb google-genai
import os
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import userdata
from agno.agent import Agent
from agno.models.google import Gemini
from agno.tools.duckdb import DuckDbTools
from IPython.display import Image, display as ipydisplay
import matplotlib.pyplot as plt
from pathlib import Path
from agno.tools import Toolkit, tool

In [5]:
os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
SHEET_ID = "1Op6IKbc0IICuQCiRYjf94MGxXfDiE-IxGcL_lGfDw4c"
GID = 558349001
url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}"
df = pd.read_csv(url)
# display(df.head())
df.groupby("League", dropna=False).size().sort_values(ascending=False)
# print(df.shape)

,0
League,
Mexico Liga TDP,996
Hungary NB III,981
USA MLS,872
Romania Liga III,826
England National League,813
...,...
El Salvador Tercera Division,1
Emirates Cup,1
England County Cup,1


In [6]:
DB_PATH = "genbi.duckdb"
# Force-close any existing connections to the file
try:
    duckdb.close(DB_PATH)
except:
    pass
con = duckdb.connect(DB_PATH)
con.execute("CREATE OR REPLACE TABLE matches AS SELECT * FROM df")
print(con.execute("SELECT COUNT(*) FROM matches").fetchone())
con.close()

(153574,)


In [7]:
#@title Semantic context:
semantic_context = """
model: football_matches
table: matches
grain: one row per match (MatchId)
columns:
  DateID: match date as YYYYMMDD integer
  League: league/competition name, includes country prefix
  MatchId: unique match identifier
  Home: home team name
  Away: away team name
  HomeLeaguePosition: home team league position at match time (often null)
  AwayLeaguePosition: away team league position at match time (often null)
  Handicap: Asian handicap line for home team (negative = home favored)
  Score: format 'FT (HT)', e.g. '2 - 1 (1 - 0)'
  Corners: format 'home-away (HThome-HTaway)'
  DangerousAttack: format 'home-away (HThome-HTaway)'
  HomeRedCard / HomeYellowCard / AwayRedCard / AwayYellowCard: card counts
notes:
  - Score, Corners, DangerousAttack are strings; use split_part() in SQL to parse.
"""

In [8]:


CHART_DIR = Path("charts")
CHART_DIR.mkdir(exist_ok=True)

class ChartToolkit(Toolkit):
    def __init__(self):
        super().__init__(name="chart_toolkit")
        self.register(self.plot_chart)

    def plot_chart(self, chart_type: str, title: str, x_label: str,
                   y_label: str, x_data: str, y_data: str) -> str:
        """
        Create a chart from CSV data and display it.
        chart_type: "line", "bar", or "scatter"
        title: chart title
        x_label: x-axis label
        y_label: y-axis label
        x_data: comma-separated x values
        y_data: comma-separated y values
        Returns the file path of the saved chart.
        """
        xs = [s.strip() for s in x_data.split(",")]
        ys = [float(s.strip()) for s in y_data.split(",")]

        fig, ax = plt.subplots(figsize=(10, 5))
        if chart_type == "line":
            ax.plot(xs, ys, marker="o", linewidth=2)
        elif chart_type == "bar":
            ax.bar(xs, ys)
        elif chart_type == "scatter":
            ax.scatter(xs, ys)
        else:
            return f"Unsupported chart type: {chart_type}"

        ax.set_title(title)
        ax.set_xlabel(x_label)
        ax.set_ylabel(y_label)
        ax.grid(True, alpha=0.3)
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()

        filename = f"chart_{len(list(CHART_DIR.glob('*.png')))}.png"
        filepath = CHART_DIR / filename
        fig.savefig(filepath, dpi=150)
        plt.close(fig)

        return str(filepath)

In [9]:
agent = Agent(
    name="FootballMatchAnalyst",
    model=Gemini(id="gemma-4-26b-a4b-it"),
    tools=[
        DuckDbTools(db_path=DB_PATH, read_only=True),
        ChartToolkit(),
    ],
    instructions=[
        "You are a football match data analyst.",
        "The DuckDB database already contains a table called 'matches'. Never create or load new tables.",
        "Use this semantic model as ground truth for column meanings:",
        semantic_context,
        "Always write and run SQL against 'matches' to answer questions — never guess numbers.",
        "Only use the plot_chart tool when the user explicitly asks for a chart, graph, or visualization.",
        "When the user asks for a chart, first run a SQL query to get the aggregated data,",
        "then call the plot_chart tool with the results.",
        "Pass x_data and y_data as comma-separated strings.",
        "For weekly grouping use: date_trunc('week', CAST(DateID AS DATE))",
        "Give a concise plain-English answer and show the SQL you used.",

    ],
    markdown=True,
)

In [15]:
question = input("Ask a question: ")

for attempt in range(10):
    try:
        agent.print_response(question, stream=True)
        charts = sorted(Path("charts").glob("*.png"))
        if charts:
            ipydisplay(Image(filename=str(charts[-1])))
        break
    except Exception as e:
        print(f"Attempt {attempt + 1} failed: {e}")
        if attempt == 9:
            print("All 10 attempts failed. Try rephrasing the question.")
        else:
          time.sleep(2)
            #show top 30 leagues with most rows

Ask a question: Best game or team


Output()

INFO Running: WITH parsed_scores AS (                                                                              
         SELECT                                                                                                    
             Home,                                                                                                 
             Away,                                                                                                 
             CAST(split_part(split_part(Score, ' (', 1), ' - ', 1) AS INTEGER) AS home_goals,                      
             CAST(split_part(split_part(Score, ' (', 1), ' - ', 2) AS INTEGER) AS away_goals                       
         FROM matches                                                                                              
     ),                                                                                                            
     team_performance AS (                                                                                         
         SELECT Home AS team, home_goals AS goals_for, -away_goals AS goals_against, CASE WHEN home_goals >        
     away_goals THEN 1 ELSE 0 END AS win FROM parsed_scores                                                        
         UNION ALL                                                                                                 
         SELECT Away AS team, away_goals AS goals_for, -home_goals AS goals_against, CASE WHEN away_goals >        
     home_goals THEN 1 ELSE 0 END AS win FROM parsed_scores                                                        
     )                                                                                                             
     SELECT team, SUM(win) AS wins, SUM(goals_for) AS goals_scored, SUM(goals_for + goals_against) AS goal_diff    
     FROM team_performance                                                                                         
     GROUP BY team                                                                                                 
     ORDER BY wins DESC, goal_diff DESC                                                                            
     LIMIT 5